# GPU health processing on Databricks

Run **01_gpu_data_starter** first. This second notebook reads its four
evidence tables, computes time-bounded health features with Spark SQL,
evaluates transparent baseline rules, and uses Delta **MERGE** to store
repeatable snapshots for the website and diagnostic-model teammate.

The data and thresholds are synthetic. These rules flag observations;
they do not diagnose a physical cause, train a model or repair hardware.
Your deterministic-model teammate can replace the rule implementation.

1. Import this `.ipynb` into the same Databricks folder and select Serverless.
2. Run all. Defaults use `workspace.gpu_ops_hackathon`, the first run ID,
   and replay time `2026-11-14T09:07:30Z`.
3. Inspect the final active alerts and the explicit unknown/clear counts.
4. In Catalog, confirm `health_features` and `health_alerts`.

Widget parameters also work when this notebook is a Databricks Job task.
To demonstrate another replay instant, rerun with another `as_of_ts`.
Scheduling this fixed timestamp repeats the same snapshot; it does NOT
create a live stream. A simulator/collector must advance the cursor first.

Only the two derived tables are updated. Original evidence remains intact.
Rerunning the same key refreshes that snapshot rather than duplicating it.
Rules emit `active`, `clear` or `unknown` for every GPU. A clear evaluation
is not a promise that the hardware is healthy, and snapshot records are
not an incident lifecycle or evidence of an autonomous repair.


In [ ]:
from datetime import datetime, timedelta, timezone
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "gpu_ops_hackathon")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("as_of_ts", "2026-11-14T09:07:30Z")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
for value in (CATALOG, SCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):
        raise ValueError("Catalog and schema must contain letters, digits and underscores.")
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

runs = spark.sql("SELECT run_id, MIN(event_ts) AS start_ts, MAX(event_ts) AS end_ts FROM telemetry GROUP BY run_id ORDER BY run_id LIMIT 100").collect()
if not runs:
    raise ValueError("No telemetry exists. Run notebook 01 first.")
RUN_ID = dbutils.widgets.get("run_id").strip() or runs[0]["run_id"]
run = next((row for row in runs if row["run_id"] == RUN_ID), None)
if run is None:
    raise ValueError("run_id was not found in the first 100 replay runs.")
AS_OF = datetime.fromisoformat(dbutils.widgets.get("as_of_ts").replace("Z", "+00:00"))
if AS_OF.tzinfo is None:
    raise ValueError("as_of_ts must include a timezone, for example 2026-11-14T09:07:30Z.")
AS_OF = AS_OF.astimezone(timezone.utc)
# Spark TIMESTAMP values collected in this UTC session are naive datetimes.
if not run["start_ts"].replace(tzinfo=timezone.utc) <= AS_OF <= run["end_ts"].replace(tzinfo=timezone.utc):
    raise ValueError("as_of_ts is outside this run's recorded replay interval.")
WINDOW_START = AS_OF - timedelta(seconds=60)
print(f"Processing: {CATALOG}.{SCHEMA}; run_id={RUN_ID}; as_of_ts={AS_OF.isoformat()}")


## Feature pipeline

Spark SQL joins GPU inventory, GPU telemetry, workload progress and cooling
telemetry at one replay cursor. It derives power relative to each device's
configured limit, temperature headroom and recent counter/progress changes.
Missing or stale streams produce NULL/unknown. A 60-second counter window
requires at least 55 seconds of coverage, no gap over 15 seconds, and no
counter reset. Old cumulative errors do not remain active new-error alerts.

This is a bounded demo query, not an optimised production ingestion pipeline.


In [ ]:
"""Transparent demo rules over Databricks-derived features, not a trained model.

Every rule emits active/clear/unknown for a snapshot. No lifecycle or root-cause
claim is inferred, and no action is performed. The deterministic-model teammate
can replace these rules while retaining this handoff contract.
"""

from datetime import datetime, timezone
import hashlib
import json

RULE_VERSION = "baseline-v1"
RULE_IDS = ("telemetry_missing", "thermal_limit", "power_limit", "new_ecc_errors", "workload_stall", "cooling_flow_low")


def evaluate_features(row: dict) -> list[dict]:
    if row["rule_version"] != RULE_VERSION:
        raise ValueError("Feature rule_version must match the implemented rule version.")
    alerts = []

    def emit(rule_id, condition, severity, summary, fields):
        timestamp = row["as_of_ts"]
        parsed = timestamp if isinstance(timestamp, datetime) else datetime.fromisoformat(str(timestamp).replace("Z", "+00:00"))
        # Spark collects TIMESTAMP as a naive datetime in the notebook's UTC session.
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        canonical_ts = parsed.astimezone(timezone.utc).isoformat()
        identity = "|".join((str(row["run_id"]), canonical_ts, row["gpu_id"], rule_id, RULE_VERSION))
        evidence = {field: row.get(field) for field in fields}
        alerts.append({
            "alert_id": hashlib.sha256(identity.encode()).hexdigest(),
            "run_id": row["run_id"], "as_of_ts": timestamp, "gpu_id": row["gpu_id"],
            "rack_id": row["rack_id"], "vendor": row["vendor"],
            "rule_id": rule_id, "rule_version": RULE_VERSION,
            "status": "unknown" if condition is None else "active" if condition else "clear",
            "severity": severity, "summary": summary,
            "evidence_json": json.dumps(evidence, sort_keys=True, default=str, allow_nan=False),
        })

    observed = bool(row["telemetry_is_observed"])
    emit("telemetry_missing", not observed, "warning", "GPU telemetry unavailable or older than 15 seconds",
         ("telemetry_ts", "telemetry_age_seconds", "telemetry_is_observed"))
    thermal = None
    if observed and row.get("temp_c") is not None and row.get("temperature_limit_c") is not None:
        thermal = row["temp_c"] >= row["temperature_limit_c"]
    emit("thermal_limit", thermal, "critical", "GPU temperature reaches its configured synthetic limit",
         ("telemetry_ts", "temp_c", "temperature_limit_c", "temperature_headroom_c", "throttled"))
    power = None if row.get("power_ratio") is None else row["power_ratio"] > 1.0
    emit("power_limit", power, "warning", "GPU power exceeds its configured synthetic limit",
         ("telemetry_ts", "power_w", "power_limit_w", "power_ratio"))
    ecc = None if row.get("ecc_delta_60s") is None else row["ecc_delta_60s"] > 0
    emit("new_ecc_errors", ecc, "critical", "Uncorrected error counter increased in the recent complete window",
         ("telemetry_ts", "ecc_delta_60s", "ecc_window_seconds"))
    stalled = None
    if row["workload_is_fresh"]:
        if row.get("job_state") == "idle":
            stalled = False
        elif row.get("job_state") == "running" and row.get("progress_delta_60s") is not None:
            stalled = row["progress_delta_60s"] == 0
    emit("workload_stall", stalled, "warning", "Running job made no progress in the recent complete window",
         ("workload_ts", "job_id", "job_state", "progress_delta_60s", "progress_window_seconds"))
    cooling = None if row.get("flow_pct") is None else row["flow_pct"] < 50.0
    emit("cooling_flow_low", cooling, "warning", "Cooling-zone flow is below the illustrative 50 percent threshold",
         ("cooling_ts", "cooling_zone_id", "flow_pct", "inlet_temp_c"))
    return alerts


In [ ]:
FEATURE_SQL = "-- Databricks processing step, not a diagnosis. All reads stop at the replay cursor.\n-- Bind run_id STRING, as_of_ts TIMESTAMP, window_start_ts TIMESTAMP, rule_version STRING.\nWITH t_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM telemetry WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), t_window AS (\n  SELECT *,\n    LAG(ecc_uncorrected_total) OVER (PARTITION BY gpu_id ORDER BY event_ts) AS prior_ecc,\n    LAG(event_ts) OVER (PARTITION BY gpu_id ORDER BY event_ts) AS prior_ts\n  FROM telemetry\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n), t_stats AS (\n  SELECT gpu_id,\n    unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts)) AS window_seconds,\n    MAX(ecc_uncorrected_total) - MIN(ecc_uncorrected_total) AS counter_delta,\n    COUNT(*) AS samples, COUNT(ecc_uncorrected_total) AS counter_samples,\n    SUM(CASE WHEN heartbeat_ok = TRUE THEN 1 ELSE 0 END) AS observed_samples,\n    MAX(CASE WHEN ecc_uncorrected_total < prior_ecc THEN 1 ELSE 0 END) AS counter_reset,\n    MAX(unix_timestamp(event_ts) - unix_timestamp(prior_ts)) AS max_gap\n  FROM t_window GROUP BY gpu_id\n), w_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY gpu_id ORDER BY event_ts DESC) AS rn\n  FROM workloads WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), w_window AS (\n  SELECT *,\n    LAG(progress_units) OVER (PARTITION BY gpu_id, job_id ORDER BY event_ts) AS prior_progress,\n    LAG(event_ts) OVER (PARTITION BY gpu_id, job_id ORDER BY event_ts) AS prior_ts\n  FROM workloads\n  WHERE run_id = :run_id AND event_ts >= :window_start_ts AND event_ts <= :as_of_ts\n), w_stats AS (\n  SELECT gpu_id, job_id,\n    unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts)) AS window_seconds,\n    MAX(progress_units) - MIN(progress_units) AS progress_delta,\n    COUNT(*) AS samples, COUNT(progress_units) AS progress_samples,\n    SUM(CASE WHEN job_state = 'running' THEN 1 ELSE 0 END) AS running_samples,\n    MAX(CASE WHEN progress_units < prior_progress THEN 1 ELSE 0 END) AS counter_reset,\n    MAX(unix_timestamp(event_ts) - unix_timestamp(prior_ts)) AS max_gap\n  FROM w_window GROUP BY gpu_id, job_id\n), c_latest AS (\n  SELECT *, ROW_NUMBER() OVER (PARTITION BY cooling_zone_id ORDER BY event_ts DESC) AS rn\n  FROM cooling WHERE run_id = :run_id AND event_ts <= :as_of_ts\n), joined AS (\n  SELECT i.*, t.event_ts AS telemetry_ts, t.heartbeat_ok,\n    t.power_w AS raw_power_w, t.gpu_util_pct AS raw_util, t.temp_c AS raw_temp_c,\n    t.throttled AS raw_throttled,\n    CASE WHEN t.heartbeat_ok = TRUE\n      AND unix_timestamp(:as_of_ts) - unix_timestamp(t.event_ts) <= 15\n      THEN TRUE ELSE FALSE END AS telemetry_is_observed,\n    ts.window_seconds AS ecc_window_seconds, ts.counter_delta,\n    CASE WHEN ts.samples >= 2 AND ts.samples = ts.counter_samples\n      AND ts.samples = ts.observed_samples AND ts.counter_reset = 0\n      AND ts.window_seconds >= 55 AND ts.max_gap <= 15\n      THEN TRUE ELSE FALSE END AS ecc_window_complete,\n    w.event_ts AS workload_ts, w.job_id, w.job_state,\n    CASE WHEN w.event_ts IS NOT NULL\n      AND unix_timestamp(:as_of_ts) - unix_timestamp(w.event_ts) <= 15\n      THEN TRUE ELSE FALSE END AS workload_is_fresh,\n    ws.window_seconds AS progress_window_seconds, ws.progress_delta,\n    CASE WHEN ws.samples >= 2 AND ws.samples = ws.progress_samples\n      AND ws.samples = ws.running_samples AND ws.counter_reset = 0\n      AND ws.window_seconds >= 55 AND ws.max_gap <= 15\n      THEN TRUE ELSE FALSE END AS progress_window_complete,\n    c.event_ts AS cooling_ts, c.inlet_temp_c AS raw_inlet_temp_c, c.flow_pct AS raw_flow_pct,\n    CASE WHEN c.event_ts IS NOT NULL\n      AND unix_timestamp(:as_of_ts) - unix_timestamp(c.event_ts) <= 15\n      THEN TRUE ELSE FALSE END AS cooling_is_fresh\n  FROM inventory i\n  LEFT JOIN t_latest t ON i.gpu_id = t.gpu_id AND t.rn = 1\n  LEFT JOIN t_stats ts ON i.gpu_id = ts.gpu_id\n  LEFT JOIN w_latest w ON i.gpu_id = w.gpu_id AND w.rn = 1\n  LEFT JOIN w_stats ws ON i.gpu_id = ws.gpu_id AND w.job_id = ws.job_id\n  LEFT JOIN c_latest c ON i.cooling_zone_id = c.cooling_zone_id AND c.rn = 1\n)\nSELECT :run_id AS run_id, :as_of_ts AS as_of_ts, :rule_version AS rule_version,\n  gpu_id, rack_id, vendor, cooling_zone_id,\n  telemetry_ts, unix_timestamp(:as_of_ts) - unix_timestamp(telemetry_ts) AS telemetry_age_seconds,\n  telemetry_is_observed,\n  CASE WHEN telemetry_is_observed THEN raw_power_w END AS power_w,\n  CASE WHEN telemetry_is_observed THEN raw_util END AS gpu_util_pct,\n  CASE WHEN telemetry_is_observed THEN raw_temp_c END AS temp_c,\n  CASE WHEN telemetry_is_observed THEN raw_throttled END AS throttled,\n  power_limit_w, temperature_limit_c,\n  CASE WHEN telemetry_is_observed AND power_limit_w > 0 THEN raw_power_w / power_limit_w END AS power_ratio,\n  CASE WHEN telemetry_is_observed THEN temperature_limit_c - raw_temp_c END AS temperature_headroom_c,\n  CASE WHEN telemetry_is_observed AND ecc_window_complete THEN counter_delta END AS ecc_delta_60s,\n  ecc_window_seconds, workload_ts, workload_is_fresh, job_id, job_state,\n  CASE WHEN workload_is_fresh AND progress_window_complete THEN progress_delta END AS progress_delta_60s,\n  progress_window_seconds, cooling_ts, cooling_is_fresh,\n  CASE WHEN cooling_is_fresh THEN raw_inlet_temp_c END AS inlet_temp_c,\n  CASE WHEN cooling_is_fresh THEN raw_flow_pct END AS flow_pct\nFROM joined\nORDER BY gpu_id;\n"
features = spark.sql(FEATURE_SQL, args={"run_id": RUN_ID, "as_of_ts": AS_OF, "window_start_ts": WINDOW_START, "rule_version": RULE_VERSION})
feature_rows = features.limit(25).collect()
if not feature_rows or len(feature_rows) > 24:
    raise ValueError("Expected 1–24 GPUs for this bounded demo. Increase backend result limits before scaling the fixture.")
if len({row['gpu_id'] for row in feature_rows}) != len(feature_rows):
    raise ValueError("Duplicate GPU IDs in feature output.")
display(features)


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

alert_fields = ["alert_id", "run_id", "as_of_ts", "gpu_id", "rack_id", "vendor", "rule_id", "rule_version", "status", "severity", "summary", "evidence_json"]
alert_schema = StructType([StructField(name, TimestampType() if name == "as_of_ts" else StringType(), False) for name in alert_fields])
alert_rows = [alert for row in feature_rows for alert in evaluate_features(row.asDict(recursive=True))]
if len(alert_rows) != len(feature_rows) * len(RULE_IDS):
    raise ValueError("Missing rule evaluations.")
if len({row["alert_id"] for row in alert_rows}) != len(alert_rows):
    raise ValueError("Duplicate alert IDs.")
alerts = spark.createDataFrame(alert_rows, schema=alert_schema)


In [ ]:
from uuid import uuid4

# Each Delta MERGE is atomic. These two tables are separate transactions.
# API consumers read health_alerts, whose entire snapshot is one MERGE.
def merge_snapshot(frame, table_name, key_fields):
    qualified = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    view = "gpu_health_" + uuid4().hex
    frame.createOrReplaceTempView(view)
    try:
        spark.sql(f"CREATE TABLE IF NOT EXISTS {qualified} USING DELTA AS SELECT * FROM `{view}` WHERE FALSE").collect()
        existing = spark.table(qualified)
        if sorted(existing.dtypes) != sorted(frame.dtypes):
            raise ValueError(f"Existing {table_name} schema differs. Use a new derived-table name/version; do not overwrite source data.")
        condition = " AND ".join(f"target.`{key}` = source.`{key}`" for key in key_fields)
        spark.sql(f"MERGE INTO {qualified} AS target USING `{view}` AS source ON {condition} WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *").collect()
    finally:
        spark.catalog.dropTempView(view)

feature_keys = ["run_id", "as_of_ts", "gpu_id", "rule_version"]
merge_snapshot(features, "health_features", feature_keys)
merge_snapshot(alerts, "health_alerts", feature_keys + ["rule_id"])
print("Stored health_features and health_alerts. Existing evidence tables were not changed.")


In [ ]:
parameters = {"run_id": RUN_ID, "as_of_ts": AS_OF, "rule_version": RULE_VERSION}
persisted_features = spark.sql("SELECT * FROM health_features WHERE run_id = :run_id AND as_of_ts = :as_of_ts AND rule_version = :rule_version", args=parameters)
persisted_alerts = spark.sql("SELECT * FROM health_alerts WHERE run_id = :run_id AND as_of_ts = :as_of_ts AND rule_version = :rule_version", args=parameters)
if persisted_features.count() != len(feature_rows) or persisted_alerts.count() != len(alert_rows):
    raise ValueError("Persisted snapshot row counts differ from this calculation.")
print(f"Validated {len(feature_rows)} features and {len(alert_rows)} rule evaluations.")
display(persisted_alerts.groupBy("status", "severity").count().orderBy("status", "severity"))
display(persisted_alerts.where("status = 'active'").orderBy("gpu_id", "rule_id"))
print(f"Website/model replay selection: run_id={RUN_ID}, as_of_ts={AS_OF.isoformat()}")


## Connect the website and model

Use the exact run ID and timestamp printed above to read `health_alerts`
through the local backend. An empty result means this snapshot has not been
processed; it does not mean that zero problems were detected. `clear` and
`unknown` rows allow the website to distinguish those outcomes explicitly.

Give active observations, their `evidence_json`, GPU/rack IDs and the fixed
replay cursor to the investigating model. It can retrieve the corresponding
GPU history and cooling/workload context. Its output should state the evidence,
uncertainty and recommended action. The baseline alerts are not model diagnoses.

To use a Databricks Job, add this notebook as a notebook task and pass the
widget parameters `catalog`, `schema`, `run_id`, `as_of_ts`. Start with a manual
run. A changing replay cursor or real ingestion must be supplied before a
repeating job can honestly be described as continuous monitoring.
